# Zebrafish hematopoiesis: clean GraphVelo start

**Analysis question**

Starting from experimentally observed developmental time (`hpf`) and
spliced/unspliced counts, infer directed and weighted relationships among
zebrafish blood-cell states, then fit exploratory ODE trajectories from HSPC
states toward supported hematopoietic branches.

This notebook deliberately does **not**:

- infer or use pseudotime;
- use scVelo or CellRank;
- treat UMAP arrows as the principal result;
- assume that cluster 23 represents the complete hematopoietic system;
- reuse pre-existing velocity, moments, neighbors, PCA, or UMAP results;
- force unsupported blood lineages into the final network.

The notebook is organized as gated stages. Read every gate before continuing.

## 0. Reproducible configuration

Edit paths only in the next cell. `INPUT_H5AD` should be the annotated
Zebrahub velocity object that retains raw `spliced` and `unspliced` layers.
The large source object is read in backed mode first; only the blood-cell
subset is copied into memory.

In [12]:
from pathlib import Path
import json
import platform
import re
import warnings

import anndata as ad
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
import scanpy as sc
import scipy
import scipy.sparse as sp
import seaborn as sns

SEED = 0
np.random.seed(SEED)

# ---- EDIT THESE PATHS ----
PROJECT_DIR = Path.home() / "dynamo" / "blood_graphvelo_clean"
INPUT_H5AD = Path.home() / "dynamo" / "zebrahub_full_velocity_annotated.h5ad"

# Set this only if the official fine annotation has a separate column name.
# The notebook will also inspect several known candidate columns.
PREFERRED_FINE_LABEL = None

RESULTS_DIR = PROJECT_DIR / "results"
CHECKPOINT_DIR = PROJECT_DIR / "checkpoints"
FIGURE_DIR = RESULTS_DIR / "figures"
for path in (RESULTS_DIR, CHECKPOINT_DIR, FIGURE_DIR):
    path.mkdir(parents=True, exist_ok=True)

N_HVG = 2000
N_PCS = 50
N_NEIGHBORS = 30
MIN_CELLS_PER_SUBTYPE = 30
TIME_TOLERANCE_HPF = 3.0

print("Python:", platform.python_version())
print("AnnData:", ad.__version__)
print("Scanpy:", sc.__version__)
print("SciPy:", scipy.__version__)
print("Input:", INPUT_H5AD)

Python: 3.11.15
AnnData: 0.10.9
Scanpy: 1.11.5
SciPy: 1.16.3
Input: /net/dali/home/mscbio/jis322/dynamo/zebrahub_full_velocity_annotated.h5ad


/tmp/ipykernel_426530/2938198587.py:42: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('scanpy')` instead
  print("Scanpy:", sc.__version__)


## 1. Source audit and clean-room rules

This gate verifies the real inputs. Existing derived results are listed only
to document what will be ignored. The analysis restarts from raw
`spliced + unspliced` counts and official annotation.

In [13]:
assert INPUT_H5AD.exists(), f"File not found: {INPUT_H5AD}"
source = ad.read_h5ad(INPUT_H5AD, backed="r")

print("Shape:", source.shape)
print("Raw layers:", list(source.layers.keys()))
print("obs columns:", source.obs.columns.tolist())
print("Existing obsm (ignored):", list(source.obsm.keys()))
print("Existing obsp (ignored):", list(source.obsp.keys()))

required_layers = {"spliced", "unspliced"}
missing_layers = required_layers.difference(source.layers.keys())
assert not missing_layers, f"Missing required layers: {sorted(missing_layers)}"
assert source.obs_names.is_unique, "Cell identifiers are not unique."

ignored_derived = {
    "velocity", "velocity_u", "fit_t", "fit_tau", "Ms", "Mu",
    "variance_velocity",
}
print("Old derived layers present but not reused:",
      sorted(ignored_derived.intersection(source.layers.keys())))

Shape: (120800, 3000)
Raw layers: ['Ms', 'Mu', 'counts', 'fit_t', 'fit_tau', 'fit_tau_', 'matrix', 'spliced', 'unspliced', 'variance_velocity', 'velocity', 'velocity_u']
obs columns: ['clusters', 'seurat_clusters', 'timepoint', 'fish', 'initial_size_spliced', 'initial_size_unspliced', 'initial_size', 'n_counts', 'velocity_self_transition', 'velocity_length', 'velocity_confidence', 'velocity_confidence_transition', 'official_fish', 'official_developmental_stage', 'official_timepoint', 'official_timepoint_cluster', 'official_zfa_class', 'official_zfa_id', 'annotation_status', 'hpf']
Existing obsm (ignored): ['X_pca', 'X_umap', 'velocity_umap']
Existing obsp (ignored): ['connectivities', 'distances']
Old derived layers present but not reused: ['Ms', 'Mu', 'fit_t', 'fit_tau', 'variance_velocity', 'velocity', 'velocity_u']


## 2. Resolve official metadata

The earlier merge established the expected fields, but this notebook verifies
them rather than assuming them. Unmatched cells remain in the master H5AD but
are excluded from this annotation-dependent working set.

In [14]:
def first_existing(columns, candidates, required=True):
    for candidate in candidates:
        if candidate and candidate in columns:
            return candidate
    if required:
        raise KeyError(f"None of these columns exists: {candidates}")
    return None

obs_columns = source.obs.columns
HPF_COL = first_existing(obs_columns, ["hpf", "hpf_numeric"])
FISH_COL = first_existing(obs_columns, ["official_fish", "fish"])
COARSE_COL = first_existing(
    obs_columns,
    ["official_zfa_class", "zebrafish_anatomy_ontology_class"],
)
CLUSTER_COL = first_existing(obs_columns, ["clusters", "seurat_clusters"])
ANNOTATION_STATUS_COL = first_existing(
    obs_columns, ["annotation_status"], required=False
)

fine_candidates = [
    PREFERRED_FINE_LABEL,
    "official_cell_type",
    "cell_type",
    "celltype",
    "annotation",
    "official_timepoint_cluster",
    "timepoint_cluster",
]
FINE_COL = first_existing(obs_columns, fine_candidates)

resolved = {
    "hpf": HPF_COL,
    "fish": FISH_COL,
    "coarse_anatomy": COARSE_COL,
    "cluster": CLUSTER_COL,
    "fine_label_candidate": FINE_COL,
    "annotation_status": ANNOTATION_STATUS_COL,
}
display(pd.Series(resolved, name="resolved_column"))

hpf                                            hpf
fish                                 official_fish
coarse_anatomy                  official_zfa_class
cluster                                   clusters
fine_label_candidate    official_timepoint_cluster
annotation_status                annotation_status
Name: resolved_column, dtype: object

## 3. Blood-cell inventory (no cluster-number shortcut)

Candidate blood cells are selected using official hematopoietic anatomy plus
fine-label keywords. Hemogenic endothelium is included because it can precede
HSPC emergence. The resulting inventory must be reviewed before subtype
assignment.

In [15]:
obs = source.obs.copy()
matched = np.ones(len(obs), dtype=bool)
if ANNOTATION_STATUS_COL is not None:
    matched = obs[ANNOTATION_STATUS_COL].astype(str).eq("matched").to_numpy()

coarse_text = obs[COARSE_COL].astype(str).str.lower()
fine_text = obs[FINE_COL].astype(str).str.lower()

blood_keywords = [
    "hematopo", "blood", "eryth", "myelo", "neutroph", "granul",
    "monocyte", "macroph", "lymph", "t cell", "b cell", "natural killer",
    "thromb", "megakaryo", "hspc", "stem cell", "progenitor",
    "hemogenic", "endothelial-to-hematopoietic",
]
keyword_pattern = "|".join(re.escape(x) for x in blood_keywords)

candidate_mask = (
    matched
    & (
        coarse_text.str.contains("hematopo", na=False).to_numpy()
        | fine_text.str.contains(keyword_pattern, regex=True, na=False).to_numpy()
    )
)
candidate_ids = obs.index[candidate_mask]
print("Candidate blood/hemogenic cells:", len(candidate_ids))
assert len(candidate_ids) > 0, "No blood-cell candidates were found."

inventory = (
    obs.loc[candidate_ids]
    .assign(
        hpf_numeric=lambda x: pd.to_numeric(x[HPF_COL], errors="coerce"),
        fine_label=lambda x: x[FINE_COL].astype(str),
        coarse_label=lambda x: x[COARSE_COL].astype(str),
        cluster_label=lambda x: x[CLUSTER_COL].astype(str),
        fish_label=lambda x: x[FISH_COL].astype(str),
    )
    .groupby(
        ["fine_label", "coarse_label", "cluster_label", "hpf_numeric"],
        dropna=False,
        observed=True,
    )
    .agg(n_cells=("fish_label", "size"), n_fish=("fish_label", "nunique"))
    .reset_index()
    .sort_values(["fine_label", "hpf_numeric", "cluster_label"])
)
inventory.to_csv(RESULTS_DIR / "blood_cell_inventory.csv", index=False)
display(inventory.head(100))

Candidate blood/hemogenic cells: 5444


,fine_label,coarse_label,cluster_label,hpf_numeric,n_cells,n_fish
0,0,hematopoietic_system,39,48.0,1,1
1,1,hematopoietic_system,19,120.0,1,1
2,10,hematopoietic_system,24,240.0,1,1
3,11,hematopoietic_system,23,240.0,1,1
7,12,hematopoietic_system,25,10.0,1,1
...,...,...,...,...,...,...
96,41,hematopoietic_system,39,48.0,40,3
95,41,hematopoietic_system,24,240.0,2,2
97,42,hematopoietic_system,23,240.0,2,2
98,42,hematopoietic_system,24,240.0,1,1


In [16]:
inventory_summary = (
    inventory.groupby("fine_label", observed=True)
    .agg(
        n_cells=("n_cells", "sum"),
        first_hpf=("hpf_numeric", "min"),
        last_hpf=("hpf_numeric", "max"),
        n_clusters=("cluster_label", "nunique"),
    )
    .sort_values("n_cells", ascending=False)
)
display(inventory_summary)
print(
    "\nGATE 1: Review this table. If FINE_COL is only a numerical "
    "timepoint-cluster rather than a biological label, stop and merge the "
    "appropriate Zebrahub fine annotation before continuing."
)

,n_cells,first_hpf,last_hpf,n_clusters
fine_label,,,,
2,941,24.0,240.0,2
17,606,120.0,120.0,2
15,604,16.0,240.0,5
9,553,240.0,240.0,2
29,374,24.0,240.0,3
28,321,19.0,120.0,4
27,304,24.0,120.0,4
20,277,19.0,240.0,4
22,273,12.0,48.0,3



GATE 1: Review this table. If FINE_COL is only a numerical timepoint-cluster rather than a biological label, stop and merge the appropriate Zebrahub fine annotation before continuing.


## 4. Create a fresh in-memory blood object

Only cell IDs selected above are copied. All inherited dimensional reductions,
graphs, velocities, model parameters, and raw snapshots are discarded.

In [17]:
source.shape

(120800, 3000)

In [18]:
len(candidate_ids)

5444

In [19]:
for layer in["spliced", "unspliced"]:
    x = source.layers[layer]
    print(layer, type(x), x.shape, x.dtype)


spliced <class 'scipy.sparse._csr.csr_matrix'> (120800, 3000) float32
unspliced <class 'scipy.sparse._csr.csr_matrix'> (120800, 3000) float32


In [20]:
# Convert the candidate-cell Boolean mask to row positions
row_idx = np.flatnonzero(candidate_mask)

print("Selected blood-cell candidates:", len(row_idx))
print("Original number of genes:", source.n_vars)

# Read only the required raw count layers
S = source.layers["spliced"][row_idx, :]
U = source.layers["unspliced"][row_idx, :]

# Convert to standard CSR matrices
S = sp.csr_matrix(S)
U = sp.csr_matrix(U)

expected_shape = (len(row_idx), source.n_vars)

print("Spliced subset shape:", S.shape)
print("Unspliced subset shape:", U.shape)
print("Expected shape:", expected_shape)

assert S.shape == expected_shape, (
    f"Unexpected spliced shape: {S.shape}; "
    f"expected {expected_shape}"
)
assert U.shape == expected_shape, (
    f"Unexpected unspliced shape: {U.shape}; "
    f"expected {expected_shape}"
)

# Copy only the corresponding metadata
selected_obs = source.obs.iloc[row_idx].copy()
selected_var = source.var.copy()

# The backed source can now be safely closed
source.file.close()
del source

# Total raw transcript counts -. main matrix part
counts = (S + U).tocsr()

# Construct a genuinely clean AnnData object
blood_clean = ad.AnnData(
    X=counts.copy(),
    obs=selected_obs,
    var=selected_var,
)

blood_clean.layers["counts"] = counts.copy()
blood_clean.layers["spliced"] = S.copy()
blood_clean.layers["unspliced"] = U.copy()

# Remove genes with zero total counts in the blood-cell subset
gene_totals = np.asarray(
    blood_clean.layers["counts"].sum(axis=0)
).ravel()

print("Zero-count genes:", int((gene_totals == 0).sum()))

blood_clean = blood_clean[:, gene_totals > 0].copy()

# Standardize essential metadata
blood_clean.obs["hpf"] = pd.to_numeric(
    blood_clean.obs[HPF_COL],
    errors="coerce",
)

blood_clean.obs["fish_id"] = (
    blood_clean.obs[FISH_COL].astype(str)
)

blood_clean.obs["source_fine_label"] = (
    blood_clean.obs[FINE_COL].astype(str)
)

assert blood_clean.obs["hpf"].notna().all(), (
    "Some candidate cells lack numeric hpf."
)
assert blood_clean.obs_names.is_unique

print("\nClean blood-cell AnnData:")
print(blood_clean)
print("Layers:", list(blood_clean.layers.keys()))
print("obsm:", list(blood_clean.obsm.keys()))
print("obsp:", list(blood_clean.obsp.keys()))

Selected blood-cell candidates: 5444
Original number of genes: 3000
Spliced subset shape: (5444, 3000)
Unspliced subset shape: (5444, 3000)
Expected shape: (5444, 3000)
Zero-count genes: 14

Clean blood-cell AnnData:
AnnData object with n_obs × n_vars = 5444 × 2986
    obs: 'clusters', 'seurat_clusters', 'timepoint', 'fish', 'initial_size_spliced', 'initial_size_unspliced', 'initial_size', 'n_counts', 'velocity_self_transition', 'velocity_length', 'velocity_confidence', 'velocity_confidence_transition', 'official_fish', 'official_developmental_stage', 'official_timepoint', 'official_timepoint_cluster', 'official_zfa_class', 'official_zfa_id', 'annotation_status', 'hpf', 'fish_id', 'source_fine_label'
    var: 'n_cells', 'highly_variable', 'means', 'dispersions', 'dispersions_norm', 'mean', 'std', 'gene_count_corr', 'velocity_gamma', 'velocity_qreg_ratio', 'velocity_r2', 'velocity_genes', 'fit_alpha', 'fit_beta', 'fit_gamma', 'fit_t_', 'fit_scaling', 'fit_std_u', 'fit_std_s', 'fit_likel

## 5. Provisional blood-subtype mapping

This is a transparent keyword mapping, not a claim that annotation is perfect.
`unresolved` is retained for review but excluded from transition modeling.
Edit `SUBTYPE_RULES` after inspecting the official labels. Rules are applied
top-to-bottom, so specific mature states should precede broad progenitor terms.

In [21]:
# check annotation first
print("FINE_COL =", FINE_COL)
print("Number of unique labels =", blood_clean.obs[FINE_COL].nunique())

fine_label_counts = (
    blood_clean.obs[FINE_COL]
    .astype(str)
    .value_counts(dropna=False)
    .rename_axis("official_label")
    .reset_index(name="n_cells")
)

display(fine_label_counts.head(200))

FINE_COL = official_timepoint_cluster
Number of unique labels = 38


,official_label,n_cells
0,2,941
1,17,606
2,15,604
3,9,553
4,29,374
5,28,321
6,27,304
7,20,277
8,22,273
9,31,246


In [22]:
label_inventory = (
    blood_clean.obs
    .assign(
        fine_label=blood_clean.obs[FINE_COL].astype(str),
        coarse_label=blood_clean.obs[COARSE_COL].astype(str),
        cluster_label=blood_clean.obs[CLUSTER_COL].astype(str),
    )
    .groupby(
        ["fine_label", "coarse_label", "cluster_label"],
        observed=True,
        dropna=False,
    )
    .size()
    .rename("n_cells")
    .reset_index()
    .sort_values("n_cells", ascending=False)
)

display(label_inventory.head(300))

,fine_label,coarse_label,cluster_label,n_cells
25,2,hematopoietic_system,23,940
18,17,hematopoietic_system,24,577
89,9,hematopoietic_system,23,552
13,15,hematopoietic_system,24,439
50,28,hematopoietic_system,23,282
...,...,...,...,...
84,6,hematopoietic_system,23,1
86,7,hematopoietic_system,24,1
87,7,hematopoietic_system,39,1
88,8,hematopoietic_system,26,1


In [23]:
timepoint_cluster_inventory = (
    blood_clean.obs
    .assign(
        hpf_numeric=pd.to_numeric(
            blood_clean.obs[HPF_COL],
            errors="coerce",
        ),
        timepoint_cluster=(
            blood_clean.obs["official_timepoint_cluster"]
            .astype(str)
        ),
        integrated_cluster=(
            blood_clean.obs[CLUSTER_COL]
            .astype(str)
        ),
    )
    .groupby(
        [
            "hpf_numeric",
            "timepoint_cluster",
            "integrated_cluster",
        ],
        observed=True,
        dropna=False,
    )
    .size()
    .rename("n_cells")
    .reset_index()
    .sort_values(
        ["hpf_numeric", "n_cells"],
        ascending=[True, False],
    )
)

display(timepoint_cluster_inventory.head(300))

,hpf_numeric,timepoint_cluster,integrated_cluster,n_cells
0,10.0,12,25,1
1,10.0,12,35,1
2,10.0,12,37,1
4,12.0,22,39,18
3,12.0,21,39,1
...,...,...,...,...
105,240.0,48,23,1
106,240.0,51,23,1
108,240.0,7,23,1
109,240.0,7,24,1


In [24]:
display(
    pd.crosstab(
        blood_clean.obs["hpf"],
        blood_clean.obs["official_timepoint_cluster"],
    )
)

official_timepoint_cluster,0,1,10,11,12,14,15,16,17,19,...,41,42,43,48,5,51,6,7,8,9
hpf,,,,,,,,,,,,,,,,,,,,,
10.0,0,0,0,0,3,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
12.0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
14.0,0,0,0,0,57,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
16.0,0,0,0,0,0,0,133,0,0,0,...,0,0,0,0,0,0,0,0,0,0
19.0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
24.0,0,0,0,0,0,1,0,2,0,0,...,0,0,0,0,0,0,0,0,1,0
48.0,1,0,0,0,0,0,1,0,0,0,...,53,0,0,0,0,0,0,0,0,0
72.0,0,0,0,0,0,0,0,0,0,31,...,0,0,0,0,0,0,0,0,0,0
120.0,0,1,0,0,1,0,1,0,606,1,...,0,0,1,0,2,0,1,1,0,0


In [25]:
def assign_subtype(label):
    text = str(label).lower()
    for subtype, patterns in SUBTYPE_RULES:
        if any(re.search(pattern, text) for pattern in patterns):
            return subtype
    return "unresolved"

blood_clean.obs["blood_subtype"] = (
    blood_clean.obs["source_fine_label"].map(assign_subtype).astype("category")
)

subtype_by_label = pd.crosstab(
    blood_clean.obs["source_fine_label"],
    blood_clean.obs["blood_subtype"],
)
display(subtype_by_label)

subtype_counts = blood_clean.obs["blood_subtype"].value_counts()
display(subtype_counts)
subtype_counts.rename("n_cells").to_csv(
    RESULTS_DIR / "provisional_blood_subtype_counts.csv"
)

NameError: name 'SUBTYPE_RULES' is not defined

In [ ]:
# download linage data to detail annotations of diff blood cells
LINEAGE_H5AD = (
    Path.home()
    / "dynamo"
    / "zf_atlas_hematopoetic_endothelial_v4_release.h5ad"
)

lineage = ad.read_h5ad(
    LINEAGE_H5AD,
    backed="r",
)

print("Lineage shape:", lineage.shape)
print("\nLineage obs columns:")

for column in lineage.obs.columns:
    print(column)

In [ ]:
candidate_annotation_columns = []

for column in lineage.obs.columns:
    name = column.lower()

    if any(
        keyword in name
        for keyword in [
            "annotation",
            "cell_type",
            "celltype",
            "label",
            "lineage",
            "zfa",
            "cluster",
        ]
    ):
        candidate_annotation_columns.append(column)

print("Candidate annotation columns:")
print(candidate_annotation_columns)

for column in candidate_annotation_columns:
    print("COLUMN:", column)

    print(
        lineage.obs[column]
        .astype(str)
        .value_counts(dropna=False)
        .head(100)
        .to_string()
    )

In [ ]:
overlap = blood_clean.obs_names.isin(
    lineage.obs_names
)

print(
    "Blood cells matching lineage object:",
    int(overlap.sum()),
    "/",
    blood_clean.n_obs,
)

print(
    "Match fraction:",
    f"{overlap.mean():.2%}",
)

In [ ]:
print("Velocity IDs:")
print(blood_clean.obs_names[:10].tolist())

print("\nLineage IDs:")
print(lineage.obs_names[:10].tolist())

In [ ]:
# fine annotation
annotation_column_summary = []

for column in lineage.obs.columns:
    values = lineage.obs[column].astype(str)

    annotation_column_summary.append({
        "column": column,
        "n_unique": values.nunique(dropna=False),
        "examples": " | ".join(
            values.drop_duplicates().head(8).tolist()
        ),
    })

annotation_column_summary = (
    pd.DataFrame(annotation_column_summary)
    .sort_values(["n_unique", "column"])
    .reset_index(drop=True)
)

display(annotation_column_summary)

In [ ]:
lineage_annotation_counts = (
    lineage.obs["annotation"]
    .astype(str)
    .value_counts(dropna=False)
    .rename_axis("official_fine_cell_type")
    .reset_index(name="n_cells")
)

display(lineage_annotation_counts)
# annotation

In [ ]:
lineage_annotation_time = pd.crosstab(
    lineage.obs["annotation"].astype(str),
    lineage.obs["timepoint"].astype(str),
)

display(lineage_annotation_time)
# annotation x time

In [ ]:
fine_annotation = (
    lineage.obs[["annotation"]]
    .copy()
    .rename(
        columns={
            "annotation": "official_fine_cell_type"
        }
    )
)

blood_clean.obs["official_fine_cell_type"] = (
    fine_annotation["official_fine_cell_type"]
    .reindex(blood_clean.obs_names)
)

blood_clean.obs["fine_annotation_status"] = np.where(
    blood_clean.obs["official_fine_cell_type"].notna(),
    "matched",
    "unmatched",
)

FINE_COL = "official_fine_cell_type"

print("Fine annotation column:", FINE_COL)

print(
    "\nFine annotation matching:"
)

print(
    blood_clean.obs["fine_annotation_status"]
    .value_counts(dropna=False)
)

print(
    "\nOfficial fine blood-cell types:"
)

print(
    blood_clean.obs[FINE_COL]
    .astype(str)
    .value_counts(dropna=False)
    .to_string()
)

In [ ]:
lineage.file.close()
del lineage

In [ ]:
display(pd.crosstab(blood_clean.obs[FINE_COL], blood_clean.obe["hfp"], margins=True))

In [ ]:
# merged annotation with ori_data 

## 6. Marker-based subtype validation

Official labels are checked against known zebrafish marker panels. Absence of a
marker from the retained 3,000-gene matrix is reported rather than interpreted
as biological absence.

In [ ]:
MARKERS = {
    "hemogenic_endothelium": ["kdrl", "kdr", "etv2", "runx1", "gata2b"],
    "hspc": ["runx1", "myb", "gata2b", "tal1", "lmo2", "cd34"],
    "erythroid_progenitor": ["gata1a", "klf1", "tal1", "alas2"],
    "erythrocyte": ["hbbe1.1", "hbbe3", "hbae1.1", "hbae3", "sptb", "alas2"],
    "myeloid_progenitor": ["spi1b", "cebpa"],
    "neutrophil": ["mpx", "lyz", "csf3r"],
    "monocyte_macrophage": ["mpeg1.1", "mfap4", "csf1ra", "lyz"],
    "thrombocyte": ["itga2b", "mpl"],
    "lymphoid_progenitor": ["ikzf1", "rag1"],
    "t_cell": ["lck", "cd3d", "cd3e", "rag1"],
    "b_cell": ["cd79a", "pax5", "ighm"],
    "nk_like": ["nkl.1", "nitr9", "prf1"],
}

present_markers = sorted({
    gene for genes in MARKERS.values() for gene in genes
    if gene in blood_clean.var_names
})
missing_markers = sorted({
    gene for genes in MARKERS.values() for gene in genes
    if gene not in blood_clean.var_names
})
print("Present marker genes:", present_markers)
print("Absent from retained matrix:", missing_markers)
assert present_markers, "None of the marker genes is present in var_names."

marker_data = blood_clean[:, present_markers].copy()
sc.pp.normalize_total(marker_data, target_sum=1e4)
sc.pp.log1p(marker_data)
marker_frame = pd.DataFrame(
    marker_data.X.toarray() if sp.issparse(marker_data.X) else marker_data.X,
    index=marker_data.obs_names,
    columns=present_markers,
)
marker_frame["blood_subtype"] = blood_clean.obs["blood_subtype"].astype(str)
marker_mean = marker_frame.groupby("blood_subtype").mean()

plt.figure(figsize=(max(10, len(present_markers) * 0.45), 6))
sns.heatmap(
    marker_mean.apply(lambda x: (x - x.mean()) / (x.std() + 1e-8), axis=0),
    cmap="vlag", center=0,
)
plt.title("Marker validation by provisional blood subtype (gene-wise z-score)")
plt.tight_layout()
plt.savefig(FIGURE_DIR / "blood_cell_subtype_validation.pdf")
plt.show()

In [ ]:
subtype_time_fish = (
    blood_clean.obs.assign(
        blood_subtype=blood_clean.obs["blood_subtype"].astype(str)
    )
    .groupby(["blood_subtype", "hpf", "fish_id"], observed=True)
    .size()
    .rename("n_cells")
    .reset_index()
)
display(subtype_time_fish)

supported_subtypes = subtype_counts[
    (subtype_counts >= MIN_CELLS_PER_SUBTYPE)
    & (subtype_counts.index.astype(str) != "unresolved")
].index.astype(str).tolist()

print("Subtypes meeting minimum cell count:", supported_subtypes)
assert "hspc" in supported_subtypes, (
    "No supported HSPC state. Do not start ODE trajectories until HSPC "
    "annotation/marker evidence is resolved."
)
assert len(supported_subtypes) >= 2, "Fewer than two supported blood subtypes."
print(
    "\nGATE 2: Continue only after the label-to-subtype table and marker "
    "heatmap are biologically defensible."
)

## 7. Restrict to validated subtypes and preprocess expression space

PCA is the principal state space. UMAP is intentionally absent from the main
pipeline. Raw count layers remain unchanged.

In [ ]:
model_mask = blood_clean.obs["blood_subtype"].astype(str).isin(supported_subtypes)
adata = blood_clean[model_mask].copy()
adata.obs["blood_subtype"] = (
    adata.obs["blood_subtype"].cat.remove_unused_categories()
)

sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)
sc.pp.highly_variable_genes(
    adata,
    n_top_genes=min(N_HVG, adata.n_vars),
    flavor="seurat",
)
sc.tl.pca(
    adata,
    n_comps=min(N_PCS, adata.n_vars - 1, adata.n_obs - 1),
    use_highly_variable=True,
    svd_solver="arpack",
    random_state=SEED,
)
sc.pp.neighbors(
    adata,
    n_neighbors=N_NEIGHBORS,
    n_pcs=adata.obsm["X_pca"].shape[1],
    use_rep="X_pca",
    random_state=SEED,
)

print(adata)
print("PCA:", adata.obsm["X_pca"].shape)

In [ ]:
pcs = adata.obsm["X_pca"]
fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))

p = axes[0].scatter(
    pcs[:, 0], pcs[:, 1], c=adata.obs["hpf"], cmap="viridis",
    s=8, alpha=0.7, rasterized=True,
)
axes[0].set(title="PCA state space by experimental hpf", xlabel="PC1", ylabel="PC2")
fig.colorbar(p, ax=axes[0], label="hpf")

for subtype in adata.obs["blood_subtype"].cat.categories:
    mask = adata.obs["blood_subtype"].astype(str).eq(str(subtype)).to_numpy()
    axes[1].scatter(
        pcs[mask, 0], pcs[mask, 1], s=8, alpha=0.65,
        label=str(subtype), rasterized=True,
    )
axes[1].set(title="PCA state space by blood subtype", xlabel="PC1", ylabel="PC2")
axes[1].legend(bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=8)
plt.tight_layout()
plt.savefig(FIGURE_DIR / "blood_pca_state_space.pdf")
plt.show()

## 8. Audit graph continuity across real time

This diagnostic prevents overclaiming a continuous developmental trajectory
when the neighbor graph contains mostly within-stage edges.

In [ ]:
C = (adata.obsp["connectivities"] > 0).astype(float).tocsr()
C_coo = C.tocoo()
hpf = adata.obs["hpf"].to_numpy(float)
edge_audit = pd.DataFrame({
    "source_hpf": hpf[C_coo.row],
    "target_hpf": hpf[C_coo.col],
})
edge_table = pd.crosstab(
    edge_audit["source_hpf"],
    edge_audit["target_hpf"],
    normalize="index",
)
display(edge_table.round(3))

same_hpf_fraction = np.mean(
    edge_audit["source_hpf"].to_numpy() == edge_audit["target_hpf"].to_numpy()
)
print(f"Same-hpf kNN edges: {same_hpf_fraction:.1%}")
if same_hpf_fraction > 0.90:
    warnings.warn(
        "More than 90% of kNN edges remain within the same hpf. Interpret "
        "velocity as local state dynamics; do not claim a single continuous "
        "trajectory spanning all sampled stages."
    )

## 9. Dynamo moments and stochastic velocity

This reproduces only the validated parts of the earlier pilot. It creates
fresh Dynamo-compatible layers from raw spliced/unspliced counts and uses the
current PCA neighbor graph.

In [ ]:
import dynamo as dyn

adata.layers["X_spliced"] = adata.layers["spliced"].copy()
adata.layers["X_unspliced"] = adata.layers["unspliced"].copy()
adata.var["use_for_pca"] = True
adata.var["pass_basic_filter"] = True
adata.var["use_for_dynamics"] = True

A = (adata.obsp["connectivities"] > 0).astype(float).tocsr()
row_sums = np.asarray(A.sum(axis=1)).ravel()
assert np.all(row_sums > 0), "Neighbor graph contains isolated cells."
moments_conn = sp.diags(1.0 / row_sums) @ A

all_genes = adata.var_names.tolist()
dyn.tl.moments(
    adata,
    genes=all_genes,
    conn=moments_conn,
    normalize=False,
    use_gaussian_kernel=False,
    layers=["X_spliced", "X_unspliced"],
    n_neighbors=N_NEIGHBORS,
)
assert "M_s" in adata.layers and "M_u" in adata.layers

In [ ]:
dyn.tl.dynamics(
    adata,
    filter_gene_mode="no",
    use_smoothed=True,
    assumption_mRNA="ss",
    model="stochastic",
    est_method="gmm",
    log_unnormalized=False,
    re_smooth=False,
    del_2nd_moments=False,
)
assert "velocity_S" in adata.layers
print("Dynamo dynamics complete.")

## 10. Velocity-gene quality control

The thresholds match the successful pilot:
`gamma > 0`, `gamma_r2 >= 0.01`, finite velocity in at least 95% of cells,
and spliced/unspliced detection in at least 25 cells. The number of passing
genes is data-dependent and is not hard-coded to the pilot's 272 genes.

In [ ]:
vel_names = list(adata.uns["vel_params_names"])
vel_params = pd.DataFrame(
    adata.varm["vel_params"],
    index=adata.var_names,
    columns=vel_names,
)

V = adata.layers["velocity_S"]
V_dense = V.toarray() if sp.issparse(V) else np.asarray(V)

def detection_count(matrix):
    return np.asarray((matrix > 0).sum(axis=0)).ravel()

qc = pd.DataFrame(index=adata.var_names)
qc["gamma"] = pd.to_numeric(vel_params["gamma"], errors="coerce")
qc["gamma_r2"] = pd.to_numeric(vel_params["gamma_r2"], errors="coerce")
qc["velocity_finite_fraction"] = np.isfinite(V_dense).mean(axis=0)
qc["spliced_detected_cells"] = detection_count(adata.layers["spliced"])
qc["unspliced_detected_cells"] = detection_count(adata.layers["unspliced"])

valid_velocity_gene = (
    qc["gamma"].gt(0)
    & qc["gamma_r2"].ge(0.01)
    & qc["velocity_finite_fraction"].ge(0.95)
    & qc["spliced_detected_cells"].ge(25)
    & qc["unspliced_detected_cells"].ge(25)
)
adata.var["velocity_qc_pass"] = valid_velocity_gene.to_numpy()
transition_genes = adata.var_names[valid_velocity_gene].tolist()
qc.to_csv(RESULTS_DIR / "velocity_gene_qc.csv")

print("Velocity-QC genes:", len(transition_genes))
assert len(transition_genes) >= 50, (
    "Too few velocity-QC genes. Inspect dynamics and subtype composition "
    "before GraphVelo."
)

## 11. GraphVelo in expression and PCA state spaces

If the installed GraphVelo source still uses SciPy sparse `.A`, replace those
package-internal occurrences with `.toarray()` once. Do not monkey-patch the
scientific data object.

In [ ]:
from graphvelo.graph_velocity import GraphVelo

gene_mask = adata.var_names.isin(transition_genes)
X_train = adata.layers["M_s"][:, gene_mask]
V_train = adata.layers["velocity_S"][:, gene_mask]
assert np.isfinite(
    X_train.data if sp.issparse(X_train) else np.asarray(X_train)
).all()
assert np.isfinite(
    V_train.data if sp.issparse(V_train) else np.asarray(V_train)
).all()

gv = GraphVelo(
    adata,
    gene_subset=transition_genes,
    xkey="M_s",
    vkey="velocity_S",
)
gv.train()

adata.layers["velocity_gv"] = gv.project_velocity(adata.layers["M_s"])
adata.obsm["gv_pca"] = gv.project_velocity(adata.obsm["X_pca"])
gv.write_to_adata(adata, key="graphvelo")

assert adata.obsm["gv_pca"].shape == adata.obsm["X_pca"].shape
assert np.isfinite(adata.obsm["gv_pca"]).all()
print("GraphVelo training and PCA projection complete.")

In [ ]:
V_input = np.asarray(gv.V)
V_fit = np.asarray(gv.project_velocity(gv.X))
input_norm = np.linalg.norm(V_input, axis=1)
fit_norm = np.linalg.norm(V_fit, axis=1)
denom = input_norm * fit_norm
fit_cosine = np.divide(
    np.sum(V_input * V_fit, axis=1),
    denom,
    out=np.full(len(denom), np.nan),
    where=denom > 0,
)
print(pd.Series(fit_cosine).describe())
if np.nanmedian(fit_cosine) <= 0:
    raise RuntimeError(
        "Median GraphVelo fit cosine is non-positive. Stop before interpreting "
        "directions or weights."
    )

## 12. Cell-level transition matrix and real-time constraint

Dynamo provides the neighbor transition matrix; GraphVelo refines state-space
velocities. We construct the cell transition matrix from GraphVelo PCA
velocities using neighbor displacement cosine similarity, retain positive
support, and remove edges that move backward by more than the allowed
experimental-time tolerance.

This is a real-time constraint, not pseudotime inference.

In [ ]:
from sklearn.preprocessing import normalize

X_pca = np.asarray(adata.obsm["X_pca"])
V_pca = np.asarray(adata.obsm["gv_pca"])
distances = adata.obsp["distances"].tocsr()

rows, cols = distances.nonzero()
displacement = X_pca[cols] - X_pca[rows]
disp_norm = np.linalg.norm(displacement, axis=1)
vel_norm = np.linalg.norm(V_pca[rows], axis=1)
denom = disp_norm * vel_norm
cosine = np.divide(
    np.sum(displacement * V_pca[rows], axis=1),
    denom,
    out=np.zeros_like(denom),
    where=denom > 0,
)

# Positive cosine support; exp sharpens direction without inventing new edges.
edge_weight = np.where(cosine > 0, np.exp(5.0 * cosine), 0.0)
T_raw = sp.csr_matrix(
    (edge_weight, (rows, cols)),
    shape=(adata.n_obs, adata.n_obs),
)
T_raw.eliminate_zeros()
T_raw = normalize(T_raw, norm="l1", axis=1)

hpf = adata.obs["hpf"].to_numpy(float)
T_coo = T_raw.tocoo()
forward_time = hpf[T_coo.col] >= (hpf[T_coo.row] - TIME_TOLERANCE_HPF)
T_time = sp.csr_matrix(
    (
        T_coo.data[forward_time],
        (T_coo.row[forward_time], T_coo.col[forward_time]),
    ),
    shape=T_raw.shape,
)
T_time.eliminate_zeros()
T_time = normalize(T_time, norm="l1", axis=1)
adata.obsp["graphvelo_time_constrained_transition"] = T_time

valid_rows = np.asarray(T_time.sum(axis=1)).ravel() > 0
print("Cells with outgoing constrained transition:", valid_rows.sum(), "/", adata.n_obs)
assert valid_rows.mean() > 0.80, "Too many cells lost all outgoing transitions."

## 13. Aggregate to directed blood-subtype weights

For source subtype \(A\) and target subtype \(B\):

\[
W_{Aightarrow B} =
rac{\sum_{i\in A}\sum_{j\in B}T_{ij}}
{\sum_{i\in A}\sum_jT_{ij}}.
\]

The table also reports the unconstrained forward-time fraction and weighted
mean change in hpf. Self-transitions are retained in the CSV but may be hidden
from the lineage figure.

In [ ]:
subtypes = adata.obs["blood_subtype"].astype(str).to_numpy()
unique_subtypes = sorted(pd.unique(subtypes))

T_raw_coo = T_raw.tocoo()
raw_edges = pd.DataFrame({
    "source": subtypes[T_raw_coo.row],
    "target": subtypes[T_raw_coo.col],
    "weight": T_raw_coo.data,
    "delta_hpf": hpf[T_raw_coo.col] - hpf[T_raw_coo.row],
})

T_time_coo = T_time.tocoo()
time_edges = pd.DataFrame({
    "source": subtypes[T_time_coo.row],
    "target": subtypes[T_time_coo.col],
    "weight": T_time_coo.data,
    "delta_hpf": hpf[T_time_coo.col] - hpf[T_time_coo.row],
})

raw_group = raw_edges.groupby(["source", "target"], observed=True)
raw_stats = raw_group.apply(
    lambda d: pd.Series({
        "forward_time_fraction_raw": np.average(
            d["delta_hpf"] >= -TIME_TOLERANCE_HPF,
            weights=d["weight"],
        ),
    })
)

time_group = time_edges.groupby(["source", "target"], observed=True)
weights = time_group["weight"].sum().rename("edge_mass")
mean_delta = time_group.apply(
    lambda d: np.average(d["delta_hpf"], weights=d["weight"])
).rename("mean_delta_hpf")

transition = pd.concat([weights, mean_delta, raw_stats], axis=1).reset_index()
source_mass = transition.groupby("source")["edge_mass"].transform("sum")
transition["weight"] = transition["edge_mass"] / source_mass
transition = transition.sort_values(["source", "weight"], ascending=[True, False])
transition.to_csv(RESULTS_DIR / "blood_transition_weights.csv", index=False)
display(transition)

## 14. Fish-level bootstrap confidence intervals

Fish—not individual cells—is the resampling unit. This avoids treating many
cells from one fish as independent biological replicates.

In [ ]:
rng = np.random.default_rng(SEED)
fish = adata.obs["fish_id"].astype(str).to_numpy()
unique_fish = np.unique(fish)
N_BOOT = 500

def aggregate_transition_for_cells(cell_indices):
    sub = T_time[cell_indices, :]
    records = []
    for source_name in unique_subtypes:
        local_rows = np.flatnonzero(subtypes[cell_indices] == source_name)
        if len(local_rows) == 0:
            continue
        block = sub[local_rows, :]
        masses = {
            target_name: float(block[:, subtypes == target_name].sum())
            for target_name in unique_subtypes
        }
        total = sum(masses.values())
        if total > 0:
            records.extend(
                (source_name, target_name, mass / total)
                for target_name, mass in masses.items()
            )
    return records

bootstrap_records = []
for iteration in range(N_BOOT):
    sampled_fish = rng.choice(unique_fish, size=len(unique_fish), replace=True)
    sampled_cells = np.concatenate([
        np.flatnonzero(fish == fish_id) for fish_id in sampled_fish
    ])
    for source_name, target_name, value in aggregate_transition_for_cells(sampled_cells):
        bootstrap_records.append((iteration, source_name, target_name, value))

boot = pd.DataFrame(
    bootstrap_records,
    columns=["iteration", "source", "target", "weight"],
)
ci = (
    boot.groupby(["source", "target"])["weight"]
    .quantile([0.025, 0.975])
    .unstack()
    .rename(columns={0.025: "weight_ci_low", 0.975: "weight_ci_high"})
    .reset_index()
)
transition_ci = transition.merge(ci, on=["source", "target"], how="left")
transition_ci.to_csv(RESULTS_DIR / "blood_transition_weights_bootstrap.csv", index=False)
display(transition_ci)

## 15. Main result: directed weighted blood-lineage network

Nodes are validated blood subtypes; node size is cell count; node color is
median experimental hpf; edge width is transition weight. Only cross-subtype
edges above the display threshold are plotted.

In [ ]:
DISPLAY_WEIGHT = 0.05
network_edges = transition_ci[
    (transition_ci["source"] != transition_ci["target"])
    & (transition_ci["weight"] >= DISPLAY_WEIGHT)
].copy()

G = nx.DiGraph()
node_stats = (
    adata.obs.assign(blood_subtype=adata.obs["blood_subtype"].astype(str))
    .groupby("blood_subtype", observed=True)
    .agg(n_cells=("hpf", "size"), median_hpf=("hpf", "median"))
)
for subtype, row in node_stats.iterrows():
    G.add_node(subtype, **row.to_dict())
for row in network_edges.itertuples():
    G.add_edge(row.source, row.target, weight=row.weight)

# Fixed biologically readable tiers; unsupported nodes use spring fallback.
preferred_pos = {
    "hemogenic_endothelium": (0, 2),
    "hspc": (0, 1),
    "erythroid_progenitor": (-2, 0),
    "erythrocyte": (-2, -1),
    "myeloid_progenitor": (-0.7, 0),
    "neutrophil": (-0.5, -1),
    "monocyte_macrophage": (0.4, -1),
    "lymphoid_progenitor": (1.0, 0),
    "t_cell": (0.8, -1),
    "b_cell": (1.5, -1),
    "nk_like": (2.1, -1),
    "thrombocyte": (2.3, 0),
}
pos = {node: preferred_pos[node] for node in G if node in preferred_pos}
missing_pos = [node for node in G if node not in pos]
if missing_pos:
    fallback = nx.spring_layout(G.subgraph(missing_pos), seed=SEED)
    pos.update({k: tuple(v) for k, v in fallback.items()})

fig, ax = plt.subplots(figsize=(11, 8))
node_order = list(G.nodes)
node_sizes = [250 + 15 * np.sqrt(G.nodes[n]["n_cells"]) for n in node_order]
node_colors = [G.nodes[n]["median_hpf"] for n in node_order]
drawn_nodes = nx.draw_networkx_nodes(
    G, pos, nodelist=node_order, node_size=node_sizes,
    node_color=node_colors, cmap="viridis", ax=ax,
)
nx.draw_networkx_labels(G, pos, font_size=9, ax=ax)
edge_widths = [1 + 10 * G.edges[e]["weight"] for e in G.edges]
nx.draw_networkx_edges(
    G, pos, width=edge_widths, arrows=True, arrowsize=18,
    connectionstyle="arc3,rad=0.08", alpha=0.75, ax=ax,
)
fig.colorbar(drawn_nodes, ax=ax, label="Median experimental hpf")
ax.set_title("Directed, weighted zebrafish blood-cell transition network")
ax.axis("off")
plt.tight_layout()
plt.savefig(FIGURE_DIR / "blood_lineage_network.pdf")
plt.show()

## 16. Exploratory branch-conditioned ODE trajectories in PCA space

We fit a continuous vector field

\[
rac{d\mathbf{x}}{dt}=f(\mathbf{x})
\]

from GraphVelo PCA velocities using distance-weighted nearest neighbors.
Each branch is fit only to HSPC plus the selected target lineage, which avoids
averaging incompatible fates into one vector at a branch point.

This is an exploratory vector-field integration, not proof that a cell literally
travels through PCA space at the inferred numerical time scale.

In [ ]:
from scipy.integrate import solve_ivp
from sklearn.neighbors import KNeighborsRegressor

BRANCHES = {
    "erythroid": ["hspc", "erythroid_progenitor", "erythrocyte"],
    "myeloid": ["hspc", "myeloid_progenitor", "neutrophil", "monocyte_macrophage"],
    "thrombocyte": ["hspc", "thrombocyte"],
    "lymphoid": ["hspc", "lymphoid_progenitor", "t_cell", "b_cell", "nk_like"],
}
BRANCHES = {
    name: [state for state in states if state in unique_subtypes]
    for name, states in BRANCHES.items()
}
BRANCHES = {
    name: states for name, states in BRANCHES.items()
    if "hspc" in states and len(states) >= 2
}
print("Supported ODE branches:", BRANCHES)
assert BRANCHES, "No HSPC-to-target branch is supported by current subtypes."

ODE_PCS = min(10, X_pca.shape[1])
N_STARTS = 5
T_SPAN = (0.0, 5.0)
T_EVAL = np.linspace(*T_SPAN, 150)

ode_trajectories = {}
for branch_name, states in BRANCHES.items():
    mask = np.isin(subtypes, states)
    X_branch = X_pca[mask, :ODE_PCS]
    V_branch = V_pca[mask, :ODE_PCS]

    field = KNeighborsRegressor(
        n_neighbors=min(50, len(X_branch)),
        weights="distance",
        n_jobs=-1,
    )
    field.fit(X_branch, V_branch)

    hspc_indices = np.flatnonzero(subtypes == "hspc")
    # Earliest HSPCs are biologically appropriate initial conditions.
    earliest_hpf = np.min(hpf[hspc_indices])
    starts = hspc_indices[hpf[hspc_indices] == earliest_hpf]
    if len(starts) > N_STARTS:
        starts = rng.choice(starts, size=N_STARTS, replace=False)

    branch_solutions = []
    for cell_index in starts:
        solution = solve_ivp(
            fun=lambda _, x: field.predict(x.reshape(1, -1))[0],
            t_span=T_SPAN,
            y0=X_pca[cell_index, :ODE_PCS],
            t_eval=T_EVAL,
            rtol=1e-5,
            atol=1e-7,
        )
        branch_solutions.append(solution.y.T)
    ode_trajectories[branch_name] = branch_solutions

In [ ]:
fig, axes = plt.subplots(
    1, len(ode_trajectories),
    figsize=(6 * len(ode_trajectories), 5),
    squeeze=False,
)
for ax, (branch_name, solutions) in zip(axes[0], ode_trajectories.items()):
    states = BRANCHES[branch_name]
    branch_mask = np.isin(subtypes, states)
    ax.scatter(
        X_pca[branch_mask, 0], X_pca[branch_mask, 1],
        c=hpf[branch_mask], cmap="viridis", s=6, alpha=0.25,
        rasterized=True,
    )
    for solution in solutions:
        ax.plot(solution[:, 0], solution[:, 1], color="black", lw=1.7)
        ax.scatter(solution[0, 0], solution[0, 1], c="red", s=30, zorder=3)
    ax.set(
        title=f"HSPC → {branch_name} ODE",
        xlabel="PC1", ylabel="PC2",
    )
plt.tight_layout()
plt.savefig(FIGURE_DIR / "blood_ode_trajectories_pca.pdf")
plt.show()

## 17. Human ortholog handoff

Ortholog mapping should be performed with a versioned source such as ZFIN or
Ensembl BioMart. Zebrafish paralogs must not be converted by merely uppercasing
gene symbols. This notebook exports the branch-relevant zebrafish genes and a
template for a separately provenance-tracked mapping.

In [ ]:
branch_gene_scores = pd.DataFrame(index=adata.var_names)
branch_gene_scores["velocity_qc_pass"] = adata.var["velocity_qc_pass"].astype(bool)
branch_gene_scores["mean_abs_graphvelo_velocity"] = np.mean(
    np.abs(np.asarray(adata.layers["velocity_gv"])), axis=0
)
branch_gene_scores["expression_variance"] = np.var(
    adata.layers["M_s"].toarray()
    if sp.issparse(adata.layers["M_s"])
    else np.asarray(adata.layers["M_s"]),
    axis=0,
)
branch_gene_scores = branch_gene_scores.sort_values(
    ["velocity_qc_pass", "mean_abs_graphvelo_velocity"],
    ascending=False,
)
branch_gene_scores.to_csv(RESULTS_DIR / "zebrafish_dynamic_gene_candidates.csv")

ortholog_template = pd.DataFrame(columns=[
    "zebrafish_gene",
    "human_ortholog",
    "orthology_type",
    "source_database",
    "source_release",
    "retrieval_date",
])
ortholog_template.to_csv(RESULTS_DIR / "zebrafish_human_ortholog_template.csv", index=False)
display(branch_gene_scores.head(50))

## 18. Save the auditable checkpoint and manifest

The checkpoint contains only the clean blood-cell analysis and its derived
objects. The original full dataset remains unchanged.

In [ ]:
adata.uns["clean_analysis_manifest"] = {
    "input_h5ad": str(INPUT_H5AD),
    "uses_pseudotime": False,
    "uses_scvelo": False,
    "uses_cellrank": False,
    "principal_state_space": "PCA",
    "experimental_time_column": HPF_COL,
    "fine_annotation_column": FINE_COL,
    "fish_column": FISH_COL,
    "n_neighbors": N_NEIGHBORS,
    "time_tolerance_hpf": TIME_TOLERANCE_HPF,
    "supported_subtypes": supported_subtypes,
}

checkpoint = CHECKPOINT_DIR / "blood_graphvelo_clean_checkpoint.h5ad"
adata.write_h5ad(checkpoint, compression="lzf")

manifest = {
    "checkpoint": str(checkpoint),
    "tables": sorted(str(p) for p in RESULTS_DIR.glob("*.csv")),
    "figures": sorted(str(p) for p in FIGURE_DIR.glob("*.pdf")),
}
with open(RESULTS_DIR / "manifest.json", "w") as handle:
    json.dump(manifest, handle, indent=2)

print(json.dumps(manifest, indent=2))

## Interpretation checklist

Before reporting:

1. Confirm every modeled subtype has official-label and marker support.
2. Report cell and fish counts across hpf; do not hide stage imbalance.
3. Report the same-hpf neighbor-edge fraction.
4. Report GraphVelo fit cosine and velocity-gene QC.
5. Present `blood_transition_weights_bootstrap.csv` with fish-level confidence
   intervals as the main quantitative answer.
6. Call the network “directed transition support,” not a proven lineage tree.
7. Describe ODE curves as exploratory integrations of the learned vector field.
8. Keep UMAP, pseudotime, scVelo, and CellRank out of the main claim.
9. Map zebrafish genes to human orthologs with a versioned database and retain
   one-to-many relationships.